# Сборка датасета

In [1]:
from onetrans.ext.yambda.datacookin import DataCookinYambdaRank
from onetrans.run.config import dataset_config


cookin = DataCookinYambdaRank()
train_set, test_set = cookin.run(dataset_config)

/Users/oleg/projects/OneTrans_HSE_project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from onetrans.ext.yambda.dataset import BinaryRankinArchive

listens, timestamp_test_start = cookin.cook(dataset_config)
archive = BinaryRankinArchive(listens)

In [3]:
meta = archive.meta

In [4]:
import polars as pl
train_listens = listens.filter(pl.col('timestamp') < timestamp_test_start)

In [5]:
from onetrans.run.config import DENSE_COLUMNS
train_listens_dense_million = train_listens[DENSE_COLUMNS].slice(0, 1_000_000)

In [6]:
batch = next(iter(train_set))

In [7]:
from onetrans.baselines.dcn_v2 import DCNV2

model = DCNV2(
    embedding_size=64,
    cross_layers=6,
    deep_units=[256, 128, 64],
    input_size=587,
    dense_train_df=train_listens_dense_million,
    n_bins=32,
    output_size=2,
    train_df_slice=1_000_000,
    num_albums=meta['num_albums'],
    num_artists=meta['num_artists'],
    num_users=meta['num_users'],
    num_items=meta['num_items']
)

In [8]:
import gc

del train_listens
del listens
del train_listens_dense_million
del archive
gc.collect()

564

In [8]:
import torch

def to_device(obj, device: torch.device | str):
    if isinstance(obj, torch.Tensor):
        return obj.to(device)

    elif isinstance(obj, dict):
        return {key: to_device(value, device) for key, value in obj.items()}

    elif isinstance(obj, list):
        return [to_device(item, device) for item in obj]

    elif isinstance(obj, tuple):
        return tuple(to_device(item, device) for item in obj)

    else:
        return obj

In [12]:
batch['targets']

{'is_like': tensor([False, False, False, False, False, False, False, False]),
 'is_full_play': tensor([False, False,  True, False, False, False,  True,  True])}

In [13]:
from torch import nn
from onetrans.utils.metrics import compute_pairwise_accuracy
from tqdm import tqdm


def evaluate(model, test_loader, device):
    model.eval()
    all_like_probs = []
    all_full_probs = []
    all_labels_like = []
    all_labels_full = []
    all_uids = []
    all_timestamps = []

    with torch.no_grad():
        for batch in test_loader:
            batch = to_device(batch, device)
            logits = model(batch)

            if logits.shape[1] == 1:
                probs = torch.sigmoid(logits.squeeze(-1))
                like_probs = probs
                full_probs = probs
            else:
                like_probs = torch.sigmoid(logits[:, 0])
                full_probs = torch.sigmoid(logits[:, 1])

            all_like_probs.append(like_probs.cpu())
            all_full_probs.append(full_probs.cpu())
            all_labels_like.append(batch["labels"]["is_like"].cpu())
            all_labels_full.append(batch["labels"]["is_full_play"].cpu())
            all_uids.append(batch["meta"]["uid"].cpu())
            all_timestamps.append(batch["meta"]["timestamp"].cpu())

    like_probs_np = torch.cat(all_like_probs).numpy()
    full_probs_np = torch.cat(all_full_probs).numpy()
    labels_like_np = torch.cat(all_labels_like).numpy()
    labels_full_np = torch.cat(all_labels_full).numpy()
    uids_np = torch.cat(all_uids).numpy()
    ts_np = torch.cat(all_timestamps).numpy()

    pair_like = compute_pairwise_accuracy(uids_np, ts_np, labels_like_np, like_probs_np)
    pair_full = compute_pairwise_accuracy(uids_np, ts_np, labels_full_np, full_probs_np)

    metrics = {
        "pair_accuracy_full_play": pair_full,
        "pair_accuracy_like": pair_like,
        "pair_accuracy_full_play_like_pairs": pair_full,
        "pair_accuracy_like_full_play_pairs": pair_like,
    }
    print(f"Метрики на тесте: pair_accuracy_full_play={pair_full}, pair_accuracy_like={pair_like}")
    return metrics


def train_epoch(
    model,
    train_loader,
    epoch, epochs,
    device,
    criterion, optimizer,
    train_log_every
):
    model.train()
    total_loss = 0.0
    steps = 0

    for batch in tqdm(train_loader, desc=f"Эпоха {epoch}/{epochs}", leave=False):
        batch = to_device(batch, device)

        optimizer.zero_grad()

        logits = model(batch['NS'])
        labels = batch["targets"]

        labels = {
            'is_like': labels["is_like"].float(),
            'is_full_play': labels["is_full_play"].float()
        }

        if logits.shape[1] == 1:
            loss = criterion(logits.squeeze(-1), labels["is_full_play"])
        else:
            loss_like = criterion(logits[:, 0], labels["is_like"])
            loss_full = criterion(logits[:, 1], labels["is_full_play"])
            loss = (loss_like + loss_full) / 2.0

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        steps += 1

        if steps % train_log_every == 0:
            print(f"Шаг {steps} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / steps
    print(f"Эпоха {epoch} завершена | Средний Loss: {avg_loss:.4f}")

def train_model(
    model,
    train_loader,
    test_loader,
    epochs: int = 5,
    lr: float = 1e-3,
    train_log_every: int = 100,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(1, epochs + 1):
        train_epoch(
            model,
            train_loader,
            epoch, epochs,
            device,
            criterion,
            optimizer,
            train_log_every
        )
        metrics = evaluate(model, test_loader, device,)

    return metrics

In [14]:
train_model(
    model=model,
    train_loader=train_set,
    test_loader=test_set
)

Эпоха 1/5:   0%|          | 102/818044 [00:06<11:15:13, 20.19it/s]

Шаг 100 | Loss: 0.4151


Эпоха 1/5:   0%|          | 203/818044 [00:11<11:21:07, 20.01it/s]

Шаг 200 | Loss: 0.6772


Эпоха 1/5:   0%|          | 302/818044 [00:15<9:58:26, 22.77it/s] 

Шаг 300 | Loss: 0.7915


Эпоха 1/5:   0%|          | 404/818044 [00:20<10:10:59, 22.30it/s]

Шаг 400 | Loss: 0.4606


KeyboardInterrupt: 